In [17]:
import pandas as pd
import os
from collections import defaultdict
import re
from typing import Dict, Set, List, Tuple
import glob

# ============================================================
# CONFIGURATION - MODIFY COUNTRIES AND CODES HERE
# ============================================================
COUNTRY_CONFIGS = {
    'xyz': ['44'],
    'ABC': ['61'],
    'WAX': ['33'],
    # Add more countries below as needed:
    # 'USA': ['1'],
    # 'Germany': ['49'],
    # 'Italy': ['39'],
    # 'Spain': ['34'],
    # 'Japan': ['81'],
    # 'China': ['86'],
    # 'Brazil': ['55'],
    # 'India': ['91'],
}
# ============================================================

class PhoneNumberAnalyzer:
    def __init__(self):
        # Use the global country configuration
        self.country_configs = COUNTRY_CONFIGS
        
        # Data structures for analysis
        self.all_phone_numbers = set()
        self.country_numbers = defaultdict(set)
        self.phone_file_mapping = defaultdict(set)  # {phone_number: {file1, file2, ...}}
        self.file_phone_mapping = defaultdict(set)  # {filename: {phone_numbers}}
        
    def normalize_phone_number(self, phone_str):
        """Normalize phone number by removing spaces, dots, and handling scientific notation"""
        if pd.isna(phone_str):
            return None
            
        # Convert to string and handle scientific notation
        phone_str = str(phone_str)
        if 'E+' in phone_str or 'e+' in phone_str:
            try:
                phone_str = str(int(float(phone_str)))
            except:
                pass
        
        # Remove non-digit characters except '+'
        phone_str = re.sub(r'[^\d+]', '', phone_str)
        
        # Remove leading '+' if present for consistent comparison
        phone_str = phone_str.lstrip('+')
        
        return phone_str if phone_str else None
    
    def get_country_code(self, phone_number):
        """Extract country code from phone number"""
        if not phone_number:
            return None
            
        # Check each configured country code
        for country, codes in self.country_configs.items():
            for code in codes:
                if phone_number.startswith(code):
                    return country, code
        return None, None
    
    def process_csv_file(self, filepath):
        """Process a single CSV file"""
        filename = os.path.basename(filepath)
        print(f"\nProcessing: {filename}")
        
        try:
            # Read CSV file
            df = pd.read_csv(filepath)
            
            # Check if required columns exist
            if 'From' not in df.columns or 'To' not in df.columns:
                print(f"  Warning: 'From' or 'To' columns not found in {filepath}")
                return
            
            # Set to track unique numbers in this file
            file_numbers = set()
            
            # Process each row
            for idx, row in df.iterrows():
                from_number = self.normalize_phone_number(row['From'])
                to_number = self.normalize_phone_number(row['To'])
                
                if from_number:
                    self.all_phone_numbers.add(from_number)
                    file_numbers.add(from_number)
                    # Check country code
                    country, code = self.get_country_code(from_number)
                    if country:
                        self.country_numbers[country].add(from_number)
                
                if to_number:
                    self.all_phone_numbers.add(to_number)
                    file_numbers.add(to_number)
                    # Check country code
                    country, code = self.get_country_code(to_number)
                    if country:
                        self.country_numbers[country].add(to_number)
            
            # Update mappings
            for number in file_numbers:
                self.phone_file_mapping[number].add(filename)
                self.file_phone_mapping[filename].add(number)
            
            print(f"  Processed {len(df)} rows, found {len(file_numbers)} unique numbers")
            
        except Exception as e:
            print(f"  Error processing {filepath}: {str(e)}")
    
    def find_common_phone_numbers(self):
        """Find phone numbers that appear in multiple CSV files"""
        common_numbers = {}
        
        # Find numbers that appear in more than one file
        for phone_number, files in self.phone_file_mapping.items():
            if len(files) > 1:
                common_numbers[phone_number] = files
        
        return common_numbers
    
    def analyze_files(self, file_pattern):
        """Analyze all CSV files matching the pattern"""
        # Print configuration
        print("="*60)
        print("PHONE NUMBER ANALYZER")
        print("="*60)
        print("\nTracking phone numbers from:")
        for country, codes in self.country_configs.items():
            print(f"  - {country}: {', '.join(codes)}")
        print("\n" + "="*60)
        
        # Find all CSV files
        csv_files = glob.glob(file_pattern)
        
        if not csv_files:
            print(f"No files found matching pattern: {file_pattern}")
            return
        
        print(f"Found {len(csv_files)} CSV files to process")
        
        # Process each file
        for filepath in csv_files:
            self.process_csv_file(filepath)
        
        # Print country-specific results
        print("\n" + "="*60)
        print("PHONE NUMBERS BY COUNTRY")
        print("="*60)
        
        for country, numbers in self.country_numbers.items():
            print(f"\n{country} (Country codes: {', '.join(self.country_configs[country])}):")
            print(f"  Total numbers found: {len(numbers)}")
            if len(numbers) <= 20:  # Show all if not too many
                for num in sorted(numbers):
                    print(f"    {num}")
            else:  # Show sample if too many
                sample = sorted(list(numbers))[:10]
                for num in sample:
                    print(f"    {num}")
                print(f"    ... and {len(numbers) - 10} more")
        
        # Find and print common phone numbers across files
        print("\n" + "="*60)
        print("PHONE NUMBERS APPEARING IN MULTIPLE FILES")
        print("="*60)
        
        common_numbers = self.find_common_phone_numbers()
        
        if not common_numbers:
            print("\nNo phone numbers found in multiple files.")
        else:
            print(f"\nFound {len(common_numbers)} phone numbers appearing in multiple files\n")
            
            # Sort by number of files (descending), then by phone number
            sorted_common = sorted(common_numbers.items(), 
                                 key=lambda x: (len(x[1]), x[0]), reverse=True)
            
            # Print all common numbers
            for phone, files in sorted_common:
                print(f"  Phone: {phone}")
                print(f"  Appears in {len(files)} files:")
                for file in sorted(files):
                    print(f"    - {file}")
                print()  # Empty line between entries
        
        # Print summary statistics
        print("\n" + "="*60)
        print("SUMMARY STATISTICS")
        print("="*60)
        print(f"Total unique phone numbers: {len(self.all_phone_numbers)}")
        print(f"Total CSV files processed: {len(self.file_phone_mapping)}")
        print(f"Numbers appearing in multiple files: {len(common_numbers)}")
        
        # File statistics
        if self.file_phone_mapping:
            file_counts = [(f, len(nums)) for f, nums in self.file_phone_mapping.items()]
            file_counts.sort(key=lambda x: x[1], reverse=True)
            print(f"\nNumbers per file:")
            for filename, count in file_counts:
                print(f"  {filename}: {count} numbers")
        
        # Save results to files
        self.save_results()
    
    def save_results(self):
        """Save analysis results to files"""
        # Save country-specific numbers
        for country, numbers in self.country_numbers.items():
            filename = f"{country}_phone_numbers.txt"
            with open(filename, 'w') as f:
                f.write(f"{country} Phone Numbers (Country code: {', '.join(self.country_configs[country])})\n")
                f.write(f"Total: {len(numbers)}\n")
                f.write("-" * 40 + "\n")
                for num in sorted(numbers):
                    f.write(f"{num}\n")
            print(f"\nSaved {country} numbers to {filename}")
        
        # Save common phone numbers (appearing in multiple files)
        common_numbers = self.find_common_phone_numbers()
        if common_numbers:
            with open("phone_numbers_in_multiple_files.txt", 'w') as f:
                f.write("Phone Numbers Appearing in Multiple CSV Files\n")
                f.write("=" * 60 + "\n")
                f.write(f"\nFound {len(common_numbers)} phone numbers appearing in multiple files\n\n")
                
                # Sort by number of files (descending), then by phone number
                sorted_common = sorted(common_numbers.items(), 
                                     key=lambda x: (len(x[1]), x[0]), reverse=True)
                
                # Write all common numbers
                for phone, files in sorted_common:
                    f.write(f"Phone: {phone}\n")
                    f.write(f"Appears in {len(files)} files:\n")
                    for file in sorted(files):
                        f.write(f"  - {file}\n")
                    f.write("\n")  # Empty line between entries
            
            print("Saved phone numbers appearing in multiple files to phone_numbers_in_multiple_files.txt")


# Main execution
if __name__ == "__main__":
    # NOTE: To change countries, modify COUNTRY_CONFIGS at the top of this file
    
    # Create analyzer instance
    analyzer = PhoneNumberAnalyzer()
    
    # Analyze all CSV files in the current directory
    analyzer.analyze_files("*.csv")

PHONE NUMBER ANALYZER

Tracking phone numbers from:
  - xyz: 44
  - ABC: 61
  - WAX: 33

Found 5 CSV files to process

Processing: 465900515.csv
  Processed 20 rows, found 21 unique numbers

Processing: 4683238017.csv
  Processed 20 rows, found 21 unique numbers

Processing: 527979219158.csv
  Processed 20 rows, found 21 unique numbers

Processing: 62391378159.csv
  Processed 20 rows, found 20 unique numbers

Processing: 9106595302.csv
  Processed 20 rows, found 21 unique numbers

PHONE NUMBERS BY COUNTRY

xyz (Country codes: 44):
  Total numbers found: 7
    4403190000000
    44075729404
    441710223
    444030462
    445069909
    4451092860
    44647881336

WAX (Country codes: 33):
  Total numbers found: 2
    3319452848
    335011036175

ABC (Country codes: 61):
  Total numbers found: 2
    611731721
    6131230000000

PHONE NUMBERS APPEARING IN MULTIPLE FILES

Found 4 phone numbers appearing in multiple files

  Phone: 2704513850
  Appears in 4 files:
    - 465900515.csv
    - 46